# HazeWaveNet - Kaggle Multi-GPU Training

Notebook n?y d?nh cho Kaggle v?i 2 GPU. D? li?u ???c ??c t? `/kaggle/input`, k?t qu?/checkpoint ghi v?o `/kaggle/working`. Cell setup t? d? th? m?c `clear/hazy` v? c?u tr?c dataset tr?n Kaggle c? th? c? th?m m?t l?p th? m?c l?ng nhau.

In [ ]:
# Cell 1 - Install dependencies and locate the repository
!pip -q install pillow matplotlib numpy tqdm thop openpyxl

import os
import sys
import subprocess
import shutil
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
WORKDIR = Path("/kaggle/working/VDT_GD2")
if not WORKDIR.exists():
    # Clone the project when Kaggle only receives this notebook.
    subprocess.run(["git", "clone", "https://github.com/Darkin72/VDT_GD2.git", str(WORKDIR)], check=True)
os.chdir(WORKDIR)
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
print("Repository:", WORKDIR)


In [ ]:
# Cell 2 - Select exactly one parallel Kaggle run: ITS or OTS
TRAIN_TARGET = "ITS"  # Change to "OTS" in the second Kaggle notebook/version.
TRAIN_TARGET = TRAIN_TARGET.upper().strip()
if TRAIN_TARGET not in {"ITS", "OTS"}:
    raise ValueError("TRAIN_TARGET must be ITS or OTS")

DATASET_ROOTS = {
    "ITS": KAGGLE_INPUT / "datasets/balraj98/indoor-training-set-its-residestandard",
    "OTS": KAGGLE_INPUT / "datasets/brunobelloni/outdoor-training-set-ots-reside",
}
RUN_CONFIG = {
    "ITS": {"lr": 2e-4, "domain": "indoor", "checkpoint": "haze_wavelet_sots_its.pt", "history": "history_sots_its.json"},
    "OTS": {"lr": 1e-4, "domain": "outdoor", "checkpoint": "haze_wavelet_sots_ots.pt", "history": "history_sots_ots.json"},
}
DATA_ROOT = DATASET_ROOTS[TRAIN_TARGET]
CONFIG = RUN_CONFIG[TRAIN_TARGET]
RUN_DIR = Path("/kaggle/working") / f"hazewavenet_{TRAIN_TARGET.lower()}"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
REPORT_DIR = RUN_DIR / "reports"
CHECKPOINT_PATH = CHECKPOINT_DIR / CONFIG["checkpoint"]
HISTORY_PATH = CHECKPOINT_DIR / CONFIG["history"]
CURVE_PATH = RUN_DIR / f"training_curve_{TRAIN_TARGET.lower()}.png"
for folder in (RUN_DIR, CHECKPOINT_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Target:", TRAIN_TARGET)
print("Dataset:", DATA_ROOT)
print("Output:", RUN_DIR)
if not DATA_ROOT.is_dir():
    raise FileNotFoundError(DATA_ROOT)
print("Folders:", sorted(path.name for path in DATA_ROOT.iterdir() if path.is_dir()))
for kind in ("clear", "hazy"):
    if not (DATA_ROOT / kind).is_dir():
        raise FileNotFoundError(DATA_ROOT / kind)


In [ ]:
# Cell 3 - Train the selected target on two GPUs
import torch
os.environ["HW_TQDM"] = "0"  # Avoid retaining thousands of tqdm redraws in Kaggle output.
os.environ["HW_TQDM_BATCH"] = "0"

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU runtime is required")
GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT < 2:
    raise RuntimeError(f"Expected 2 GPUs, found {GPU_COUNT}")
print("GPUs:", GPU_COUNT, [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)])

command = [
    sys.executable, "-u", "-m", "solution.wavelet_dehaze.train", str(DATA_ROOT),
    "--device", "cuda", "--multi-gpu", "--epochs", "10",
    "--batch-size", "64", "--num-workers", "2", "--prefetch-factor", "2",
    "--pin-memory", "--persistent-workers", "--amp",
    "--val-fraction", "0.1", "--split-seed", "42", "--size", "256",
    "--lr", str(CONFIG["lr"]), "--weight-decay", "1e-4",
    "--scheduler", "cosine", "--min-lr", "1e-6", "--augment", "--multi-scale",
    "--patience", "50", "--out", str(CHECKPOINT_PATH), "--history", str(HISTORY_PATH),
]
print("Running:", " ".join(command))
subprocess.run(command, cwd=WORKDIR, check=True)

for artifact in (CHECKPOINT_PATH, HISTORY_PATH):
    if not artifact.is_file():
        raise FileNotFoundError(artifact)
    print(f"Saved {artifact} ({artifact.stat().st_size / 1024**2:.2f} MB)")


In [ ]:
# Cell 4 - Training curves for the selected run: loss, PSNR and SSIM
import json
import matplotlib.pyplot as plt

if not HISTORY_PATH.is_file():
    raise FileNotFoundError(HISTORY_PATH)
records = json.loads(HISTORY_PATH.read_text())
if not records:
    raise RuntimeError(f"Empty training history: {HISTORY_PATH}")

epochs = [record["epoch"] for record in records]
metrics = (("train_loss", "val_loss", "Loss"), ("train_psnr", "val_psnr", "PSNR (dB)"), ("train_ssim", "val_ssim", "SSIM"))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (train_key, val_key, title) in zip(axes, metrics):
    ax.plot(epochs, [record[train_key] for record in records], label="Train", linewidth=2)
    if val_key in records[0]:
        ax.plot(epochs, [record[val_key] for record in records], "--", label="Validation", linewidth=2)
    ax.set_title(f"SOTS-{TRAIN_TARGET} - {title}")
    ax.set_xlabel("Epoch"); ax.set_ylabel(title); ax.grid(alpha=0.25); ax.legend()
plt.tight_layout()
plt.savefig(CURVE_PATH, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", CURVE_PATH)


In [ ]:
# Cell 5 - Evaluate only the SOTS split matching TRAIN_TARGET
SOTS_TEST_ROOT = KAGGLE_INPUT / "datasets/balraj98/synthetic-objective-testing-set-sots-reside"
EVAL_DATA = RUN_DIR / "evaluation_data"
EVAL_DATA.mkdir(parents=True, exist_ok=True)

def link_directory(source, target):
    source, target = Path(source), Path(target)
    if not source.is_dir():
        raise FileNotFoundError(source)
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.symlink_to(source, target_is_directory=True)

# Create all required evaluator slots, but populate only the selected synthetic domain.
for dataset_name in ("I-HAZE", "O-HAZY"):
    for kind in ("clear", "hazy"):
        (EVAL_DATA / dataset_name / "test" / kind).mkdir(parents=True, exist_ok=True)
sots_target = EVAL_DATA / "Synthetic Objective Testing Set (SOTS) [RESIDE]"
for domain in ("indoor", "outdoor"):
    for kind in ("clear", "hazy"):
        target = sots_target / domain / "test" / kind
        if domain == CONFIG["domain"]:
            link_directory(SOTS_TEST_ROOT / domain / kind, target)
        else:
            target.mkdir(parents=True, exist_ok=True)

# Four checkpoint arguments are required by the evaluator; empty dataset slots are never inferred.
command = [
    sys.executable, "-u", "-m", "solution.wavelet_dehaze.evaluate",
    "--data-root", str(EVAL_DATA), "--output-dir", str(REPORT_DIR),
    "--checkpoint-i-haze", str(CHECKPOINT_PATH), "--checkpoint-o-hazy", str(CHECKPOINT_PATH),
    "--checkpoint-sots-its", str(CHECKPOINT_PATH), "--checkpoint-sots-ots", str(CHECKPOINT_PATH),
    "--device", "cuda", "--max-side", "1024",
]
print("Running:", " ".join(command))
subprocess.run(command, cwd=WORKDIR, check=True)


In [ ]:
# Cell 6 - Add params/FLOPs and verify this run's artifacts
import pandas as pd
from thop import profile
from solution.wavelet_dehaze.model import HazeWaveNet

state = torch.load(CHECKPOINT_PATH, map_location="cuda")
complexity_rows = []
for side, purpose in ((256, "Training"), (1024, "Evaluation upper bound")):
    model = HazeWaveNet().cuda().eval()
    model.load_state_dict(state.get("model", state))
    params = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    dummy = torch.zeros(1, 3, side, side, device="cuda")
    with torch.inference_mode():
        macs, _ = profile(model, inputs=(dummy,), verbose=False)
    complexity_rows.append({
        "Model": "HazeWaveNet", "Training target": TRAIN_TARGET, "Purpose": purpose,
        "Input": f"1x3x{side}x{side}", "Parameters": params,
        "Trainable parameters": trainable, "GMACs": macs / 1e9,
        "GFLOPs (2 x MACs)": 2 * macs / 1e9,
        "GPU count": GPU_COUNT, "GPU": torch.cuda.get_device_name(0),
    })
complexity_df = pd.DataFrame(complexity_rows)
hardware_path = REPORT_DIR / "hardware.xlsx"
if not hardware_path.is_file():
    raise FileNotFoundError(hardware_path)
with pd.ExcelWriter(hardware_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    complexity_df.to_excel(writer, sheet_name="Complexity", index=False)

expected = {
    "Checkpoint": CHECKPOINT_PATH,
    "Training history": HISTORY_PATH,
    "Training curves": CURVE_PATH,
    "Dataset report": REPORT_DIR / "hazewavenet_dataset.xlsx",
    "Domain report": REPORT_DIR / "hazewavenet_domain.xlsx",
    "Fog report": REPORT_DIR / "hazewavenet_fog.xlsx",
    "Hardware report": hardware_path,
}
for label, path in expected.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    print(f"{label}: {path} ({path.stat().st_size / 1024**2:.2f} MB)")
complexity_df
